# BlinkBuild Demo: Ollama on Google Colab via ngrok

**Demo-only mode (`local`).** Free Colab runtimes may terminate endpoints; never use this as the sole provider.
use this url:https://colab.research.google.com/github/SinghalSahab/BlinkBuild/blob/feat/multi-agent-engine/server/colab/ollama_demo.ipynb

### Required Colab Secrets
Add these in Colab **Secrets (key icon)**:
- `NGROK_AUTHTOKEN`: your ngrok auth token
- `OLLAMA_BEARER_TOKEN`: a random secret token (must match `OLLAMA_BEARER_TOKEN` in BlinkBuild's `.env`)

In [ ]:
# Supply your coder model name here before running (D-016: never hard-coded).
OLLAMA_MODEL = "qwen2.5-coder:14b"  # e.g. set at runtime

if not OLLAMA_MODEL.strip():
    raise ValueError("Set OLLAMA_MODEL before running this notebook (D-016).")

In [ ]:
# Install Ollama and pyngrok
!sudo apt-get update && sudo apt-get install -y pciutils zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q pyngrok

In [ ]:
import subprocess, time, urllib.request

# Start Ollama server on localhost:11434
ollama_proc = subprocess.Popen(["ollama", "serve"])
time.sleep(3)

# Pull and warm up the user-supplied model
subprocess.run(["ollama", "pull", OLLAMA_MODEL], check=True)
req = urllib.request.Request(
    "http://127.0.0.1:11434/api/generate",
    data=f'{{"model": "{OLLAMA_MODEL}", "prompt": "hi", "stream": false}}'.encode(),
    headers={"Content-Type": "application/json"},
)
with urllib.request.urlopen(req, timeout=60) as resp:
    print("Warmup status:", resp.status)

In [ ]:
import hmac, threading, http.server, urllib.request
from google.colab import userdata
from pyngrok import ngrok

NGROK_TOKEN = userdata.get("NGROK_AUTHTOKEN")
BEARER_TOKEN = userdata.get("OLLAMA_BEARER_TOKEN")
if not NGROK_TOKEN or not BEARER_TOKEN:
    raise RuntimeError("Missing NGROK_AUTHTOKEN or OLLAMA_BEARER_TOKEN in Colab Secrets")

class BearerProxyHandler(http.server.BaseHTTPRequestHandler):
    def _proxy(self, method):
        auth = self.headers.get("Authorization", "")
        expected = f"Bearer {BEARER_TOKEN}"
        if not hmac.compare_digest(auth, expected):
            self.send_response(401)
            self.end_headers()
            self.wfile.write(b'{"error": "Unauthorized"}')
            return
        length = int(self.headers.get("Content-Length", 0))
        body = self.rfile.read(length) if length > 0 else None
        req = urllib.request.Request(
            f"http://127.0.0.1:11434{self.path}",
            data=body,
            method=method,
            headers={"Content-Type": "application/json"},
        )
        with urllib.request.urlopen(req, timeout=120) as upstream:
            data = upstream.read()
            self.send_response(upstream.status)
            self.send_header("Content-Type", "application/json")
            self.end_headers()
            self.wfile.write(data)

    def do_GET(self): self._proxy("GET")
    def do_POST(self): self._proxy("POST")
    def log_message(self, *_args): pass

server = http.server.ThreadingHTTPServer(("127.0.0.1", 11435), BearerProxyHandler)
threading.Thread(target=server.serve_forever, daemon=True).start()

ngrok.set_auth_token(NGROK_TOKEN)
tunnel = ngrok.connect(11435, "http")
print("Set OLLAMA_BASE_URL to:", tunnel.public_url)